# Apple Detection with YOLOv8n — Merged Roboflow Datasets

**Datasets:**
- `paramee/apple-tiyxx` (~216 images)
- `l61l/apple-desti` (~1,270 images)

**Model:** YOLOv8n (nano, 3.2M params)
**Runtime:** GPU (T4 recommended)

## 1. Setup & Install

In [ ]:
!pip install -q --upgrade numpy ultralytics roboflow
import os
os.kill(os.getpid(), 9)

## 2. Download & Merge Datasets

In [ ]:
from roboflow import Roboflow
import shutil
from pathlib import Path
import glob

API_KEY = "evNnmWyUlNy3mu4YXnhM"
rf = Roboflow(api_key=API_KEY)

print("=== Downloading Dataset 1: paramee/apple-tiyxx ===")
project1 = rf.workspace("paramee").project("apple-tiyxx")
dataset1 = project1.versions()[0].download("yolov8")

print("\n=== Downloading Dataset 2: l61l/apple-desti ===")
project2 = rf.workspace("l61l").project("apple-desti")
dataset2 = project2.versions()[0].download("yolov8")

MERGED_DIR = Path("/content/datasets/apple_merged")
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    (MERGED_DIR / sub).mkdir(parents=True, exist_ok=True)

def merge_split(src_dir, dest_img_dir, dest_lbl_dir, prefix=""):
    for img in glob.glob(f"{src_dir}/**/*.jpg", recursive=True):
        fname = prefix + "_" + Path(img).name if prefix else Path(img).name
        shutil.copy(img, dest_img_dir / fname)
    for lbl in glob.glob(f"{src_dir}/**/*.txt", recursive=True):
        if "data.yaml" in lbl or "README" in lbl:
            continue
        fname = prefix + "_" + Path(lbl).name if prefix else Path(lbl).name
        shutil.copy(lbl, dest_lbl_dir / fname)

print("\n=== Merging train splits ===")
for d, p in [(dataset1.location, "d1"), (dataset2.location, "d2")]:
    for split in ["train", "valid", "test"]:
        src = Path(d) / split
        if src.exists():
            merge_split(str(src), MERGED_DIR / "images" / "train", MERGED_DIR / "labels" / "train", prefix=p)

print("=== Merging val splits ===")
for d, p in [(dataset1.location, "d1"), (dataset2.location, "d2")]:
    src = Path(d) / "valid"
    if src.exists():
        merge_split(str(src), MERGED_DIR / "images" / "val", MERGED_DIR / "labels" / "val", prefix=p)

yaml = f"""path: {MERGED_DIR}
train: images/train
val: images/val
nc: 1
names:
  0: apple
"""
(MERGED_DIR / "data.yaml").write_text(yaml)

train_imgs = list((MERGED_DIR / "images" / "train").glob("*.jpg"))
val_imgs = list((MERGED_DIR / "images" / "val").glob("*.jpg"))
print(f"\nTrain: {len(train_imgs)} | Val: {len(val_imgs)} | Total: {len(train_imgs) + len(val_imgs)}")

DATA_YAML = str(MERGED_DIR / "data.yaml")

## 3. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    save=True,
    device=0,
    amp=True,
    project="runs/apple_detect",
    name="yolov8n_merged_apple",
)

## 4. Validate & Visualize

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Find best.pt dynamically (handles Ultralytics path nesting)
possible_paths = [
    Path("runs/apple_detect/yolov8n_merged_apple/weights/best.pt"),
    Path("/content/runs/detect/runs/apple_detect/yolov8n_merged_apple/weights/best.pt"),
    Path("/content/runs/apple_detect/yolov8n_merged_apple/weights/best.pt"),
]

BEST_PT = None
for p in possible_paths:
    if p.exists():
        BEST_PT = str(p)
        print(f"Found: {BEST_PT}")
        break

if BEST_PT is None:
    # Auto-search
    import glob
    matches = glob.glob("/content/**/best.pt", recursive=True)
    if matches:
        BEST_PT = matches[0]
        print(f"Auto-found: {BEST_PT}")

best_model = YOLO(BEST_PT)

# Validate
metrics = best_model.val(data=DATA_YAML)
print(f"mAP@0.5:    {metrics.box.map50:.4f}")
print(f"mAP@0.5:95: {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

# Visualize random validation images
import random, glob
from PIL import Image
from IPython.display import display

val_imgs = glob.glob(f"{MERGED_DIR}/images/val/*.jpg")
for img in random.sample(val_imgs, min(5, len(val_imgs))):
    r = best_model(img, conf=0.25, verbose=False)
    display(Image.fromarray(r[0].plot()))
    print(f"{Path(img).name}: {len(r[0].boxes)} apples")

## 5. Export & Download

In [ ]:
from google.colab import files

best_model.export(format="onnx")
files.download(BEST_PT)

print("Download best.pt and rename to yolov8_apple.pt")